# Week-1 Project: Company Sales Brochure Generator:

## Objective:
    * Create a product that can generate marketing brochure about a company.

## Technology:
    * Use OpenAI API
    * One-shot prompting
    * Stram back results and show with formating.

### Importing all the necessary libraries.

In [76]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import display, Markdown, update_display
from scraping import fetch_website_contents,fetch_website_links

### Loading API key

In [2]:
load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if not api_key:
    raise ValueError("OPENAI_API_KEY environment variable is not set.")
else:
    print("The API key is loaded successfully.")

model = 'gpt-5-nano'
openai = OpenAI()

The API key is loaded successfully.


In [14]:
links = fetch_website_links("https://www.langchain.com/")
for link in links:
    if "https" in link:
        print(link)

https://interrupt.langchain.com/london
https://www.youtube.com/playlist?list=PLfaIDFEXuae3UwB1QGEjsRAr8BzCQss7s
https://academy.langchain.com/
https://www.youtube.com/@LangChain
https://docs.langchain.com/
https://luma.com/langchain?k=c
https://docs.langchain.com/
https://smith.langchain.com/
https://smith.langchain.com/
https://smith.langchain.com/
https://www.youtube.com/watch?v=cx6_tb6HCeY
https://www.langchain.com/blog/how-rippling-went-ai-native-across-every-product-in-6-months-with-deep-agents-and-langsmith
https://www.youtube.com/watch?v=UVeeNW_z068
https://www.youtube.com/watch?v=kuXtW03cZEA
https://www.youtube.com/watch?v=mxweSHetuN8
https://www.youtube.com/watch?v=EAj0HBDGqaE
https://www.langchain.com/blog/customers-monday
https://www.youtube.com/watch?v=Yy3JH6dDugc
https://www.youtube.com/watch?v=lXZb21CfeIY
https://www.langchain.com/blog/customers-servicenow
https://www.youtube.com/watch?v=py9d6zTl4Dc
https://docs.langchain.com/langsmith/prompt-context-hub
https://blog.lang

In [15]:
print(links)

['https://interrupt.langchain.com/london', '/', '/langsmith-platform', '/langsmith/engine', '/langsmith/observability', '/langsmith/evaluation', '/langsmith/deployment', '/langsmith/sandboxes', '/langsmith/llm-gateway', '/langsmith/fleet', '/oss-overview', '/deep-agents', '/langgraph', '/langchain', '/blog', '/customers', '/resources', 'https://www.youtube.com/playlist?list=PLfaIDFEXuae3UwB1QGEjsRAr8BzCQss7s', 'https://academy.langchain.com/', 'https://www.youtube.com/@LangChain', 'https://docs.langchain.com/', '/startups', 'https://luma.com/langchain?k=c', '/community', 'https://docs.langchain.com/', '/about', '/careers', '/langchain-partner-network', '/events', '/pricing', 'https://smith.langchain.com/', '/contact-sales', 'https://smith.langchain.com/', '/contact-sales', '/langsmith/engine', 'https://smith.langchain.com/', '/contact-sales', 'https://www.youtube.com/watch?v=cx6_tb6HCeY', 'https://www.langchain.com/blog/how-rippling-went-ai-native-across-every-product-in-6-months-with-

## Using gpt-5-nano to filter out unrelevant links from the list:

* Using "one-shot" prompting, we will let gpt model to convert relative links such as '/about' with "https://company.com/about".
* Makes the job easier to clear out messy links thorugh this method.

In [10]:
system_prompt = """ 
    Based on the given list of links from a website, select the most relevant links that provide information about the website's features, use cases, careers page and documnetation.
    The output should be a strucutres JSON object which follows the following fomrat:

    {
        "links:[
            {"type":"about page", "url": "https://www.example.com/about"},
            {"type":"careers page", "url": "https://www.example.com/careers"}
        ]       
    }
"""

### Function which combines the user prompt and the list of links from the website.

In [49]:
def user_prompt_links(url):
    user_prompt = f"""
    The floowing is the list of links from the given website: {url}.
    Please select the most relevant links that provides information about the website's features, use cases, blog, relevant social media page links,careers page and the documentation.
    Output should include the full https url in json format.
    Do not inlude terms of service, privacy policy or any other irrelevant links."""

    links = fetch_website_links(url)
    user_prompt = user_prompt +"\n"+ "\n".join(links)
    return user_prompt

In [32]:
prompt = user_prompt_links("https://www.langchain.com/")
print(prompt)


    The floowing is the list of links from the given website: https://www.langchain.com/.
    Please select the most relevant links that provides information about the website's features, use cases, careers page and the documentation.
    Output should include the full https url in json format.
    Do not inlude terms of service, privacy policy or any other irrelevant links.
https://interrupt.langchain.com/london
/
/langsmith-platform
/langsmith/engine
/langsmith/observability
/langsmith/evaluation
/langsmith/deployment
/langsmith/sandboxes
/langsmith/llm-gateway
/langsmith/fleet
/oss-overview
/deep-agents
/langgraph
/langchain
/blog
/customers
/resources
https://www.youtube.com/playlist?list=PLfaIDFEXuae3UwB1QGEjsRAr8BzCQss7s
https://academy.langchain.com/
https://www.youtube.com/@LangChain
https://docs.langchain.com/
/startups
https://luma.com/langchain?k=c
/community
https://docs.langchain.com/
/about
/careers
/langchain-partner-network
/events
/pricing
https://smith.langchain.com/

## Writing a function to select relevant links from the given list of links

In [50]:
def get_relevant_links(url,model):
    print(f"The given model was used: {model}")
    response = openai.chat.completions.create(
        model = model,
        messages = [
            {"role":"system", "content": system_prompt},
            {"role":"user","content": user_prompt_links(url)}
        ],
        response_format = {"type":"json_object"}
    )
    result = response.choices[0].message.content
    # converting the json object to a python dictionary
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links in the given website: {url}")
    return links



In [46]:
get_relevant_links("https://www.langchain.com/", "gpt-5-nano")

The given model was used: gpt-5-nano
Found 5 relevant links in the given website: https://www.langchain.com/


{'links': [{'type': 'features page',
   'url': 'https://www.langchain.com/oss-overview'},
  {'type': 'use cases page', 'url': 'https://www.langchain.com/customers'},
  {'type': 'blog', 'url': 'https://blog.langchain.com/'},
  {'type': 'documentation', 'url': 'https://docs.langchain.com/'},
  {'type': 'careers page', 'url': 'https://www.langchain.com/careers'}]}

## Second Step: Making the brochure.

In [51]:
def fetch_page_relevant_links(url,model):
    contents = fetch_website_contents(url)
    relevant_links = get_relevant_links(url,model)
    result = f"Website contents: {contents} \n\n Relevant links: \n"
    for link in relevant_links['links']:
        result+=f"\n\n ##Link:{link['type']}\n"
        result+=fetch_website_contents(link['url'])
    return result


In [52]:
print(fetch_page_relevant_links('https://www.langchain.com/', 'gpt-5-nano'))

The given model was used: gpt-5-nano
Found 10 relevant links in the given website: https://www.langchain.com/
Website contents: LangChain: the open agent platform to own your intelligence

Interrupt is coming to London on October 13!
Get tickets
Products
LangSmith Platform
Agent Improvement
Engine
Improve agents autonomously
Observability
See exactly what your agents are doing
Evaluation
Score and improve agent performance
Agent Infrastructure
Deployment
Ship and scale agents in production
Sandboxes
Run agent-generated code safely
LLM Gateway
Control agent model calls
No-Code Agents
Fleet
Agents for the whole company
Open Source
Agent Frameworks
deepagents
Build long-running agents for complex tasks
langgraph
Build agents with low-level control
langchain
Quick start agents with any model provider
Learn
Resources
Blog
Customer Stories
Guides
Max Agency
How-To
LangChain Academy
YouTube
Documentation
Community
LangSmith for Startups
Meetups
Community
Docs
Company
About
Careers
Partners
Ev

## Creating brochure focused system and user prompts:

In [53]:
brochure_system_prompt = """
    You are a marketing expert. Based on the given webiste contents, relevant links and the contents of those links, create
    a brochure for the website. The brochure should be structured and should include the following details:
    1. Website Overview: A brief introduction to the website, its purpose, and its target audience.
    2. Key Features: Highlight the main features of the website, explaining how they benefit users.
    3. Use Cases: Provide examples of how different users can utilize the product or service offered by the website.
    4. Careers: If applicable, include information about career opportunities, company culture, and how to apply.
    5. Documentation: Summarize the available documentation, guides, or resources that help users understand and use the website effectively.
    The brochure should be concise, engaging, and informative, making it easy for readers to understand the value of the website and its offerings."""

In [55]:
def get_brochure_user_prompt(company_name, url, model):
    user_prompt = f"""
    You are going through the details of a company called: {company_name}.
    Based on the given contents from the company's website and other relevant pages, create a short brochure of the company in markdown without code blocks.\n\n
    """

    user_prompt+= fetch_page_relevant_links(url, model)
    user_prompt = user_prompt[:5_000]
    return user_prompt

In [58]:
get_brochure_user_prompt("Langchain", "https://www.langchain.com/", "gpt-5-nano")

The given model was used: gpt-5-nano
Found 20 relevant links in the given website: https://www.langchain.com/


"\n    You are going through the details of a company called: Langchain.\n    Based on the given contents from the company's website and other relevant pages, create a short brochure of the company in markdown without code blocks.\n\n\n    Website contents: LangChain: the open agent platform to own your intelligence\n\nInterrupt is coming to London on October 13!\nGet tickets\nProducts\nLangSmith Platform\nAgent Improvement\nEngine\nImprove agents autonomously\nObservability\nSee exactly what your agents are doing\nEvaluation\nScore and improve agent performance\nAgent Infrastructure\nDeployment\nShip and scale agents in production\nSandboxes\nRun agent-generated code safely\nLLM Gateway\nControl agent model calls\nNo-Code Agents\nFleet\nAgents for the whole company\nOpen Source\nAgent Frameworks\ndeepagents\nBuild long-running agents for complex tasks\nlanggraph\nBuild agents with low-level control\nlangchain\nQuick start agents with any model provider\nLearn\nResources\nBlog\nCustome

## Defining a function for generating the brochure

In [66]:
def create_brochure(company_name, url, model):
    response = openai.chat.completions.create(
        model = model,
        messages = [
            {"role":"system","content":brochure_system_prompt},
            {"role":"user","content":get_brochure_user_prompt(company_name, url, model)}
        ],
    )

    result = response.choices[0].message.content
    display(Markdown(result))

In [67]:
create_brochure("Langchain","https://www.langchain.com/","gpt-5-nano")

The given model was used: gpt-5-nano
Found 15 relevant links in the given website: https://www.langchain.com/


# LangChain: The Open Agent Platform to Own Your Intelligence

## Website Overview
LangChain is an open platform designed to help teams build, deploy, and govern AI agents that can autonomously perform tasks, reason, and interact with users. It serves developers, product teams, startups, and enterprises looking to own their AI harness—from building and testing to observability and production deployment. The ecosystem combines open-source frameworks, tooling for evaluation and observability, and enterprise-ready deployment and governance.

## Key Features
- LangSmith Platform: A unified workflow to test, observe, and improve agent behavior with actionable insights.
- Observability: See exactly what your agents are doing, with traces and data to understand performance and failures.
- Evaluation: Score and optimize agent performance to raise reliability and consistency.
- Deployment: Ship and scale agents in production-ready infrastructure with long-running tasks and persistence.
- Sandboxes: Run agent-generated code safely to explore capabilities without risk.
- LLM Gateway: Control and manage model calls for agent interactions.
- No-Code Agents: Build useful agents without writing code, accelerating time-to-value.
- Fleet: Enable every team to build agents using natural language, empowering organization-wide automation.
- Open Source Agent Stack: Access proven frameworks to customize and extend your agent capabilities.
  - deepagents: Long-running agents for complex tasks
  - langgraph: Low-level control for advanced workflows
  - langchain: Quick-start agents with any model provider

Additional resources:
- Documentation and Guides
- LangChain Academy, YouTube, and Community events
- LangSmith for Startups, Meetups, and Case Studies

## Use Cases
- AI Engineers and Developers: Rapidly prototype, test, and deploy autonomous agents with confidence using full lifecycle tooling.
- Product Teams: Build AI-powered assistants and automated workflows that scale across teams.
- Enterprises: Govern and observe agent behavior, profile performance, and ensure security and reliability in production.
- Startups: Leverage LangSmith and the OSS stack to experiment fast and validate concepts with real-world data.
- Operations and IT: Orchestrate long-running tasks, maintain persistent state, and manage model calls across environments.

## Careers
LangChain offers opportunities across engineering, product, documentation, community, and partnerships. The company emphasizes collaboration, open-source values, and building tools that empower developers to own their intelligence. Current roles and application details are available on the Careers page.

## Documentation
LangChain provides a rich set of resources to help users understand and maximize the platform:
- Documentation site with guides and API references
- Guides and How-To content for common patterns and implementations
- LangChain Academy for structured learning
- Blog, customer stories, and case studies to illustrate real-world use
- YouTube tutorials and community-led events
- Community resources: Meetups, forums, and shared best practices

If you’re ready to accelerate your agent development, explore LangChain today, start a trial or request a demo, and join a growing community of builders who are owning their intelligence.

In [68]:
create_brochure("Langchain","https://www.langchain.com/","gpt-4.1-mini")

The given model was used: gpt-4.1-mini
Found 9 relevant links in the given website: https://www.langchain.com/


# LangChain Brochure

---

## Website Overview

**LangChain** is an open-source AI agent platform designed to empower organizations and developers to build, deploy, and manage intelligent AI agents efficiently. With a rich pre-built architecture and seamless integrations for any AI model or tool, LangChain enables users to adapt quickly in the fast-moving AI ecosystem. Its target audience includes AI developers, enterprises looking to automate with smart agents, startups, and technology teams eager to harness advanced AI capabilities without vendor lock-in.

---

## Key Features

- **LangSmith Platform**: Centralized platform to improve agents autonomously through observability, evaluation, and deployment tools.
- **Agent Improvement Engine**: Automatically enhance agent functionality by observing and scoring performance.
- **Agent Infrastructure**: Supports scalable production deployment with persistent state management, secure sandbox environments for running code, and control over model API calls.
- **No-Code Agents with Fleet**: Enables teams company-wide to build and manage AI agents using natural language without needing to write code.
- **Open Source Agent Frameworks**:
  - **deepagents**: Build long-running agents for handling complex tasks.
  - **langgraph**: Allows creation of agents with fine-grained, low-level control.
  - **langchain**: Simplifies getting started quickly with any model provider using proven templates.
- **Durable Runtime**: Agents run on LangGraph's durable runtime featuring persistence, rewind, checkpointing, and human-in-the-loop capabilities.
- **Modular and Customizable**: Easily extend agent behaviors with middleware hooks for approval workflows, data compression, or privacy management without changing core logic.
- **Broad Integrations**: Over 1000 integrations with models, tools, and databases ensure flexibility and future-proofing without vendor lock-in.
- **Observability and Evaluation**: Real-time tracing, complex query capabilities, and standardized expert reviews turn agent interaction data into actionable insights.

---

## Use Cases

- **AI Developers:** Rapidly prototype and deploy smart agents customized for customer support, data querying, or internal automation.
- **Enterprises:** Automate repetitive business processes with agents that seamlessly integrate into existing infrastructure, ensuring data privacy and compliance.
- **Startups:** Leverage no-code agent builders to empower non-technical teams and reduce time-to-market for AI-powered products.
- **Technology Teams:** Monitor and iterate on AI agents with advanced observability tools to ensure reliability and continuous improvement.
- **Research & Development:** Experiment with sophisticated agent behaviors using open frameworks like deepagents and langgraph for complex task handling.

---

## Careers

LangChain fosters a culture of innovation and open collaboration, welcoming passionate individuals eager to shape the future of AI agent technology. Career opportunities span engineering, product management, research, and community engagement roles. To join the team and contribute to a rapidly evolving AI platform, visit the Careers section on the LangChain website for current openings and application details.

---

## Documentation & Resources

LangChain offers comprehensive documentation and learning resources to help users at every skill level:

- **Official Docs:** Step-by-step guides on building agents with Python and other supported languages.
- **LangChain Academy:** Structured tutorials and courses for mastering the platform.
- **Blog & Customer Stories:** Insights and case studies demonstrating real-world applications and best practices.
- **How-To Guides & Max Agency:** Practical instructions for specific use cases.
- **YouTube Channel:** Video tutorials, product demos, and event recordings.
- **Community & Meetups:** Active forums and events to connect with other LangChain users and contributors.

For quick access, users can try LangChain with demos or explore open-source repositories to kickstart their projects immediately.

---

Empower your intelligence. Build smarter, scalable AI agents with LangChain.

Explore more and get started at [langchain.com](https://langchain.com)  
Get a demo | Join the community | Read the docs

## Using it on HuggingFace website

In [69]:
create_brochure("HuggingFace","https://huggingface.co/","gpt-4.1-mini")

The given model was used: gpt-4.1-mini
Found 25 relevant links in the given website: https://huggingface.co/


# Hugging Face Brochure

## Website Overview
Hugging Face is the leading AI community platform dedicated to building the future of machine learning. It serves as a collaborative space where developers, researchers, and organizations come together to create, share, and innovate with machine learning models, datasets, and applications. With a vast collection of over 2 million models and 500,000+ datasets, Hugging Face empowers users ranging from AI enthusiasts and data scientists to enterprises, helping them accelerate AI development and deployment.

## Key Features
- **Extensive Model Repository:** Access and share a diverse collection of models covering tasks like text generation, image-to-text, image generation, speech recognition, and more. Models are updated frequently, ensuring cutting-edge performance.
  
- **Rich Dataset Library:** Explore a variety of datasets across formats including text, image, audio, video, and tabular data supporting machine learning research and production.

- **Spaces:** Host and run AI-powered applications in a seamless environment, enabling interactive demos, prototypes, and custom AI apps.

- **Community Collaboration:** Join one of the largest AI communities to share insights, contribute to open-source projects, and participate in discussions via forums, Discord, and GitHub.

- **Enterprise Solutions:** Offers scalable enterprise support, including Hugging Face PRO, inference endpoints, dedicated infrastructure, and enterprise-grade collaboration tools for commercial AI deployments.

- **Comprehensive Documentation:** Extensive guides, API references, and tutorials available to support users at all skill levels to learn, implement, and optimize machine learning workflows.

## Use Cases
- **Researchers & Developers:** Discover pre-trained models and datasets to accelerate research, fine-tune models for specialized tasks, and share findings with the global AI community.

- **Businesses & Enterprises:** Deploy AI models at scale with enterprise support for inference endpoints and storage buckets, enabling integration in products like chatbots, image recognition, or predictive analytics.

- **Educators & Learners:** Utilize the learning resources and easy-to-access AI applications to teach machine learning concepts or experiment with state-of-the-art models interactively.

- **AI Enthusiasts & Hobbyists:** Experiment with popular AI models and create unique AI-powered applications hosted on Hugging Face Spaces, fostering creativity and innovation.

## Careers
Hugging Face values innovation, collaboration, and inclusivity in building the machine learning ecosystem. The team consists of passionate AI researchers, developers, and community builders dedicated to open science and AI democratization. Career opportunities are regularly available across roles such as ML engineering, research, developer advocacy, product management, and more. Interested candidates can visit the Hugging Face website to apply and contribute to shaping the future of AI.

## Documentation
The platform provides rich documentation covering:
- How to use the Hugging Face Hub for model and dataset management.
- Tutorials on popular machine learning tasks with libraries such as Transformers, Diffusers, and more.
- Guides for deploying models using inference endpoints and integration with enterprise platforms.
- Community tutorials and blog posts for staying updated on latest trends and use cases.
- APIs and SDK references to help developers build custom solutions.

---

**Join Hugging Face today and be part of the AI community that is transforming the future of machine learning!**

Visit: [huggingface.co](https://huggingface.co)

## Creating a stream function to display the output on the run.

In [77]:
def stream_ouput(company_name,url,model):
    stream = openai.chat.completions.create(
        model = model,
        messages = [
            {"role":"system","content":brochure_system_prompt},
            {"role":"user","content":get_brochure_user_prompt(company_name,url,model)}
        ],
        stream=True
    )
    response = ""
    display_handle = display(Markdown(""), display_id = True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response),display_id=display_handle.display_id)

In [79]:
stream_ouput("Langchain","https://www.langchain.com/","gpt-4.1-mini")

The given model was used: gpt-4.1-mini
Found 8 relevant links in the given website: https://www.langchain.com/


# LangChain Brochure

## Website Overview  
LangChain is a cutting-edge open-source AI agent platform designed to empower individuals and organizations to own and tailor their intelligence solutions. By providing an adaptable framework compatible with any AI model or tool, LangChain enables rapid agent development and seamless integration, all while avoiding vendor lock-in. The platform is ideal for developers, enterprises, startups, and AI enthusiasts seeking to build, test, deploy, and monitor intelligent agents efficiently.

## Key Features  

- **Open Source AI Agent Framework**: Build and customize AI agents quickly using pre-built architectures and extensive integrations (over 1000+), supporting any model and tool.  
- **LangSmith Platform**: Enhance your agents autonomously with tools for observability, evaluation, and improvement to optimize performance continuously.  
- **Agent Infrastructure**: Ship and scale agents in production environments with persistence, checkpointing, and long-running task support powered by LangGraph’s durable runtime.  
- **No-Code Agents with Fleet**: Enable entire organizations to build and deploy agents using natural language interfaces, making sophisticated AI accessible across teams.  
- **Safe Execution Environment (Sandboxes)**: Run agent-generated code securely without risking production environments.  
- **Customizable Middleware**: Extend agent capabilities easily with composable hooks for human-in-the-loop approval, conversation compression, sensitive data removal, and more.  
- **Observability & Evaluation Tools**: Track every step of agent behavior in detail, debug runs with step-by-step tracing, and turn data into actionable insights for score-based improvements.  
- **Durable Runtime**: Maintain agent state with rewind and checkpoint functionality to build more reliable, persistent AI solutions.  

## Use Cases  

- **Developers & AI Engineers**: Rapidly prototype and iterate on intelligent agents, leveraging templates and integration flexibility to suit varied scenarios.  
- **Enterprises**: Deploy scalable AI-powered assistants across departments using Fleet, managing agents without extensive coding.  
- **Startups**: Accelerate product innovation by integrating reliable, persistent agents with customizable behaviors for customer support, automation, or analytics.  
- **Research & Academia**: Utilize LangChain’s open-source frameworks and rich documentation to experiment with advanced AI agent designs and workflows.  
- **Organizations Needing Safe AI Execution**: Run complex agent-generated workflows safely with sandboxed code execution and controlled model calls.  

## Careers  
LangChain is a fast-growing company passionate about redefining how AI agents are built and deployed. They foster a collaborative and innovative culture, offering opportunities for professionals eager to work at the frontier of AI technology. Interested candidates can explore career openings on LangChain’s website and apply to join a team dedicated to empowering users worldwide with intelligent agent tools.

## Documentation  
LangChain offers extensive, user-friendly documentation and learning resources to help users maximize the platform’s potential:  

- **Getting Started Guides**: Step-by-step tutorials for building agents with LangChain’s Python SDK and other tools.  
- **Comprehensive API References**: Detailed specifications for core components like LangChain, LangGraph, and DeepAgents.  
- **Customer Stories and Case Studies**: Real-world examples showcasing agent applications and benefits.  
- **LangChain Academy & YouTube Channel**: Interactive and video tutorials to deepen understanding and facilitate rapid skill acquisition.  
- **Community & Meetups**: Engage with an active, global user community via forums and events for support and networking.  

---

### Experience the future of AI agent development with LangChain — build smarter, adaptable, and scalable agents your way!

**Try LangSmith | Get a Demo | Visit www.langchain.com**